# Analyse the results of a study

This notebook reads the results of the `example` study, lists the runs that are not done
and compares methods with a critical-difference diagram, paired tests and a numbers file
for the paper.

Open it with `uv sync --group notebooks`, then pick the kernel of the project environment.
The kernel starts in `notebooks/`, so every path below is relative to that folder.

Files land in `notebooks/analysis/`:

```
analysis/
  results_numbers.tex, results_numbers.csv   numbers for the text, `\res{key}` in LaTeX
  stats/        test results as CSV
  tables/       metrics and summaries as CSV
  coverage/     missing and failed runs as CSV
  plots/<category>/<format>/<name>.<format>   one folder per chosen file type
  synthetic/    everything made from the made-up demo data, with the same layout
```

## Setup

Imports first. Nothing here needs the project package.

In [ ]:
from datetime import datetime, UTC
import os
from pathlib import Path

import numpy as np
import pandas as pd
from tslab.analysis import (
    AnalysisMetric,
    AnalysisOutput,
    Caption,
    cd_diagram,
    Direction,
    FigureFormat,
    load_results,
    method_matrix,
    missing_runs,
    number_key,
    NumberKind,
    numbers_frame,
    NUMBERS_STEM,
    OutputFolder,
    pairwise_wilcoxon,
    paper_style,
    show,
    write_numbers,
)

Settings: the metric to compare, the columns that name the method and the dataset, and the
output folder. The `instance` column picks the evaluation: `evaluate[default]` is the
clean one, and the noisy ones are `evaluate[noise-std-<std>]`.

In [ ]:
METRIC = AnalysisMetric(name='accuracy', label='accuracy', direction=Direction.HIGHER_IS_BETTER)
METHOD = 'recipe'
DATASET = 'dataset'
# The clean evaluation; the noisy ones are evaluate[noise-std-<std>].
INSTANCE = 'evaluate[default]'
# The file types of every figure are chosen here.
OUTPUT = AnalysisOutput(root=Path('analysis'), formats=(FigureFormat.PDF, FigureFormat.PNG))

The project: the study and the machine profile that its runs ran on.

In [ ]:
from {{package_name}}.studies.example import study as example

PROJECT_ROOT = Path('..')
# Relative to PROJECT_ROOT.
MACHINE = Path('machines/local.toml')

## 1. Data

One row per run and evaluation, with the axes of the study as columns and one
`metric/<name>` column per metric.

In [ ]:
RESULTS = load_results(
    study=example, machine=MACHINE, project_root=PROJECT_ROOT, environment=os.environ
)
RESULTS

## 2. Coverage

Every run that is not done, or has no evaluation yet. Run `tslab study resubmit` for these.

In [ ]:
MISSING = missing_runs(results=RESULTS)
OUTPUT.save_table(frame=MISSING, folder=OutputFolder.COVERAGE, name='missing_runs')
show(blocks=[Caption(f'{len(MISSING)} rows without a finished evaluation.'), MISSING])

Sections 3 and 4 below use made-up data: the example study has one recipe and one dataset
per grid, which is too little for a comparison. Every table, figure, caption and file made
from it says "SYNTHETIC DATA", and its files go to `analysis/synthetic/`.

To analyse your own study, change the cell below: set `SYNTHETIC = False` and replace the
`SCORES = ...` line with

```python
SCORES = RESULTS[RESULTS['instance'] == INSTANCE]
```

Then check that `METHOD` and `DATASET` name columns of your results.

In [ ]:
# synthetic
SYNTHETIC = True
GENERATOR = np.random.default_rng(0)
SKILL = {'method-a': 0.9, 'method-b': 0.8, 'method-c': 0.75, 'method-d': 0.7}
DIFFICULTY = GENERATOR.normal(0, 0.05, size=10)
SCORES = pd.DataFrame(
    [
        {
            METHOD: method,
            DATASET: f'dataset-{index:02d}',
            'seed': seed,
            'instance': INSTANCE,
            METRIC.column: skill + DIFFICULTY[index] + GENERATOR.normal(0, 0.02),
        }
        for method, skill in SKILL.items()
        for index in range(10)
        for seed in range(2)
    ]
)
DEMO = OUTPUT.child(name='synthetic') if SYNTHETIC else OUTPUT
PREFIX = 'synthetic_' if SYNTHETIC else ''
MARK = '[SYNTHETIC DATA] ' if SYNTHETIC else ''

## 3. Synthetic demo data: compare the methods

Synthetic demo data. The matrix has one row per dataset and one column per method; the
seeds are averaged. Each pair of rows and columns must be unique, so the `instance` column is
filtered before this step.

In [ ]:
# synthetic
MATRIX = method_matrix(
    frame=SCORES, unit=DATASET, method=METHOD, value=METRIC.column, average_over=['seed']
)
MATRIX

Synthetic demo data. The critical-difference diagram ranks the methods on every dataset
(1 is best) and averages the ranks. The Friedman test checks whether the methods differ at
all, and a bar joins the methods that the Nemenyi critical difference cannot tell apart.

In [ ]:
# synthetic
with paper_style():
    DIAGRAM = cd_diagram(matrix=MATRIX, direction=METRIC.direction)
BLOCKS = [Caption(MARK + DIAGRAM.caption)]
if DIAGRAM.figure is not None:
    if SYNTHETIC:
        DIAGRAM.figure.text(
            0.5,
            0.5,
            'SYNTHETIC DATA',
            ha='center',
            va='center',
            rotation=20,
            fontsize=32,
            color='grey',
            alpha=0.25,
        )
    DEMO.save_figure(
        figure=DIAGRAM.figure, category='critical_difference', name=f'{PREFIX}cd_{METRIC.name}'
    )
    BLOCKS.append(DIAGRAM.figure)
show(blocks=BLOCKS)

Synthetic demo data. A paired Wilcoxon signed-rank test for every pair of methods, with
Holm-adjusted p-values and the rank-biserial correlation as the effect size.

In [ ]:
# synthetic
WILCOXON = pairwise_wilcoxon(matrix=MATRIX)
DEMO.save_table(
    frame=WILCOXON, folder=OutputFolder.STATS, name=f'{PREFIX}pairwise_wilcoxon_{METRIC.name}'
)
show(blocks=[Caption(MARK + 'Paired Wilcoxon signed-rank tests, Holm-adjusted.'), WILCOXON])

## 4. Synthetic demo data: numbers for the paper

Synthetic demo data. The best method, its average rank and the Friedman p-value, written as
LaTeX macros: `\input{results_numbers}` in the paper, then `\res{best-accuracy}`.

In [ ]:
# synthetic
NUMBERS = numbers_frame(
    records=[
        (number_key(parts=['best', METRIC.name]), str(DIAGRAM.ranks.index[0]), NumberKind.TEXT),
        (
            number_key(parts=['best', METRIC.name, 'rank']),
            float(DIAGRAM.ranks.iloc[0]),
            NumberKind.SCORE,
        ),
        (number_key(parts=['friedman', METRIC.name, 'p']), DIAGRAM.friedman.p_value, NumberKind.P),
    ]
)
PATHS = write_numbers(
    numbers=NUMBERS,
    output=DEMO,
    source='analysis.ipynb',
    now=datetime.now(tz=UTC),
    stem='synthetic_numbers' if SYNTHETIC else NUMBERS_STEM,
    note='synthetic demo values, not results' if SYNTHETIC else None,
)
NUMBERS